# 🫁 COVID-19 X-Ray Detection — CNN + ResNet50 + VGG16
**Adapted from Google Colab → VS Code Local**

---
### ⚡ One-time terminal setup (paste in VS Code terminal with myenv999 active):
```
pip install numpy==1.26.4 matplotlib==3.8.4 pandas==2.2.1 scikit-learn seaborn Pillow kaggle tensorflow==2.15.0 opencv-python
```
After install → **Restart Kernel** → **Run All**

---

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Setup: Kaggle credentials + Download dataset
# ═══════════════════════════════════════════════════════════
import os, json, pathlib, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ── Write kaggle credentials ─────────────────────────────────
CREDS = {"username": "simransihag",
         "key":      "8af325981473e165d0e0136b416cfc0c"}

kdir = pathlib.Path.home() / '.kaggle'
kdir.mkdir(parents=True, exist_ok=True)
kfile = kdir / 'kaggle.json'
kfile.write_text(json.dumps(CREDS))
kfile.chmod(0o600)
os.environ['KAGGLE_USERNAME'] = CREDS['username']
os.environ['KAGGLE_KEY']      = CREDS['key']
print('✅ Kaggle credentials ready')

# ── Download dataset ─────────────────────────────────────────
import kaggle

SLUG = 'pranavraikokte/covid19-image-dataset'
BASE = pathlib.Path('data')

if not BASE.exists() or not any(BASE.rglob('*.jpg')):
    print('⬇️  Downloading dataset (~158 MB)...')
    BASE.mkdir(parents=True, exist_ok=True)
    kaggle.api.authenticate()
    kaggle.api.dataset_download_files(SLUG, path=str(BASE),
                                      unzip=True, quiet=False)
    for z in BASE.glob('*.zip'):
        z.unlink()
    print('✅ Download complete!')
else:
    print('✅ Dataset already present — skipping download')

# ── Auto-find train/test dirs ────────────────────────────────
train_path = test_path = None
for d in sorted(BASE.rglob('*')):
    if d.is_dir() and d.name.lower() == 'train':
        kids = [c.name.lower() for c in d.iterdir() if c.is_dir()]
        if 'covid' in kids:
            train_path = d
    if d.is_dir() and d.name.lower() == 'test':
        kids = [c.name.lower() for c in d.iterdir() if c.is_dir()]
        if 'covid' in kids:
            test_path = d

if not train_path or not test_path:
    print('📁 Extracted structure:')
    for p in sorted(BASE.rglob('*')):
        if p.is_dir():
            print('  ' * len(p.relative_to(BASE).parts) + f'📂 {p.name}')
    raise RuntimeError('Cannot find train/test — see structure above')

# Use string paths (matches your original notebook style)
train_path = str(train_path)
test_path  = str(test_path)

print(f'\n📂 Train : {train_path}')
print(f'📂 Test  : {test_path}')

classes = ['Covid', 'Normal', 'Viral Pneumonia']
for c in classes:
    nt = len(os.listdir(os.path.join(train_path, c)))
    nv = len(os.listdir(os.path.join(test_path, c)))
    print(f'   {c:<20} → {nt:>4} train  |  {nv:>3} test')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Load images (same logic as your Colab notebook)
# Uses PIL instead of cv2 to avoid Windows errors
# ═══════════════════════════════════════════════════════════
import numpy as np
from PIL import Image

IMG_SIZE = 128

X_train, y_train = [], []
X_test,  y_test  = [], []

print('📂 Loading training images...')
for category in classes:
    path  = os.path.join(train_path, category)
    label = classes.index(category)
    for img_name in os.listdir(path):
        try:
            img_p = os.path.join(path, img_name)
            img   = Image.open(img_p).convert('RGB')
            img   = img.resize((IMG_SIZE, IMG_SIZE))
            X_train.append(np.array(img))
            y_train.append(label)
        except:
            pass
    print(f'  ✅ {category}: {len([y for y in y_train if y == label])} images')

print('\n📂 Loading test images...')
for category in classes:
    path  = os.path.join(test_path, category)
    label = classes.index(category)
    for img_name in os.listdir(path):
        try:
            img_p = os.path.join(path, img_name)
            img   = Image.open(img_p).convert('RGB')
            img   = img.resize((IMG_SIZE, IMG_SIZE))
            X_test.append(np.array(img))
            y_test.append(label)
        except:
            pass

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test  = np.array(X_test)
y_test  = np.array(y_test)

print(f'\n✅ Train Shape : {X_train.shape}')
print(f'✅ Test Shape  : {X_test.shape}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Visualise sample images
# ═══════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))
for i in range(6):
    plt.subplot(2, 3, i+1)
    plt.imshow(X_train[i])
    plt.title(classes[y_train[i]])
    plt.axis('off')
plt.suptitle('Sample Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Class distribution
# ═══════════════════════════════════════════════════════════
import seaborn as sns

for category in classes:
    train_count = len(os.listdir(os.path.join(train_path, category)))
    test_count  = len(os.listdir(os.path.join(test_path,  category)))
    print(f'{category}')
    print(f'  Train Images : {train_count}')
    print(f'  Test Images  : {test_count}')
    print()

class_counts = [len(os.listdir(os.path.join(train_path, c))) for c in classes]
plt.figure(figsize=(6, 4))
sns.barplot(x=classes, y=class_counts, palette=['#e74c3c','#2ecc71','#3498db'])
plt.title('Class Distribution')
plt.xlabel('Classes')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — Preprocess + split
# ═══════════════════════════════════════════════════════════
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# Normalize
X_train_norm = X_train / 255.0
X_test_norm  = X_test  / 255.0

# One-hot encode labels
y_train_cat = to_categorical(y_train, num_classes=3)
y_test_cat  = to_categorical(y_test,  num_classes=3)

# Train / validation split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_norm, y_train_cat, test_size=0.2, random_state=42
)

print('✅ Training   :', X_tr.shape)
print('✅ Validation :', X_val.shape)
print('✅ Test       :', X_test_norm.shape)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Basic CNN (your original model)
# ═══════════════════════════════════════════════════════════
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Train Basic CNN
# ═══════════════════════════════════════════════════════════
print('🚀 Training Basic CNN...')
history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    verbose=1
)

# Accuracy plot
plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'],     label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Basic CNN — Accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.tight_layout(); plt.show()

# Loss plot
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Basic CNN — Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.tight_layout(); plt.show()

test_loss, test_acc = model.evaluate(X_test_norm, y_test_cat, verbose=0)
print(f'\n✅ Basic CNN Test Accuracy: {test_acc:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — Data augmentation generators
# ═══════════════════════════════════════════════════════════
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rotation_range=20, zoom_range=0.2,
    width_shift_range=0.2, height_shift_range=0.2,
    horizontal_flip=True
)
val_datagen = ImageDataGenerator()

train_generator = train_datagen.flow(X_tr,  y_tr,  batch_size=32)
val_generator   = val_datagen.flow(X_val, y_val, batch_size=32)

print('✅ Data generators ready')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — Deep CNN (your 3-block model)
# ═══════════════════════════════════════════════════════════
from tensorflow.keras.callbacks import EarlyStopping

deep_cnn = Sequential([
    Conv2D(32,  (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),
    Conv2D(64,  (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

deep_cnn.compile(optimizer='adam',
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=3,
                           restore_best_weights=True)

print('🚀 Training Deep CNN...')
history_deep = deep_cnn.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stop],
    verbose=1
)

plt.figure(figsize=(8,5))
plt.plot(history_deep.history['accuracy'],     label='Train')
plt.plot(history_deep.history['val_accuracy'], label='Val')
plt.title('Deep CNN Accuracy'); plt.xlabel('Epoch')
plt.legend(); plt.tight_layout(); plt.show()

deep_loss, deep_acc = deep_cnn.evaluate(X_test_norm, y_test_cat, verbose=0)
print(f'\n✅ Deep CNN Test Accuracy: {deep_acc:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — VGG16 Transfer Learning
# ═══════════════════════════════════════════════════════════
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.optimizers import Adam

# Preprocess for VGG16
X_tr_vgg  = vgg_preprocess(X_tr  * 255)
X_val_vgg = vgg_preprocess(X_val * 255)
X_te_vgg  = vgg_preprocess(X_test_norm * 255)

vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(128,128,3))
for layer in vgg_base.layers:
    layer.trainable = False

vgg_model = Sequential([
    vgg_base,
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

vgg_train_gen = ImageDataGenerator(
    rotation_range=20, zoom_range=0.2,
    width_shift_range=0.2, height_shift_range=0.2,
    horizontal_flip=True
).flow(X_tr_vgg, y_tr, batch_size=32)

vgg_val_gen = ImageDataGenerator().flow(X_val_vgg, y_val, batch_size=32)

vgg_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=3,
                           restore_best_weights=True)

print('🚀 Training VGG16...')
history_vgg = vgg_model.fit(
    vgg_train_gen,
    validation_data=vgg_val_gen,
    epochs=15,
    callbacks=[early_stop],
    verbose=1
)

plt.figure(figsize=(8,5))
plt.plot(history_vgg.history['accuracy'],     label='Train')
plt.plot(history_vgg.history['val_accuracy'], label='Val')
plt.title('VGG16 Accuracy'); plt.xlabel('Epoch')
plt.legend(); plt.tight_layout(); plt.show()

vgg_loss, vgg_acc = vgg_model.evaluate(X_te_vgg, y_test_cat, verbose=0)
print(f'\n✅ VGG16 Test Accuracy: {vgg_acc:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — VGG16 Fine Tuning (unfreeze last 4 layers)
# ═══════════════════════════════════════════════════════════
for layer in vgg_base.layers[-4:]:
    layer.trainable = True

vgg_model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('🚀 Fine-tuning VGG16 (last 4 layers unfrozen)...')
history_finetune = vgg_model.fit(
    vgg_train_gen,
    validation_data=vgg_val_gen,
    epochs=10,
    callbacks=[early_stop],
    verbose=1
)

fine_loss, fine_acc = vgg_model.evaluate(X_te_vgg, y_test_cat, verbose=0)
print(f'\n✅ Fine-Tuned VGG16 Accuracy: {fine_acc:.4f}')

# Use the fine-tuned accuracy as vgg_acc
vgg_acc = fine_acc

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 12 — ResNet50 Transfer Learning
# ═══════════════════════════════════════════════════════════
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as res_preprocess
from tensorflow.keras.callbacks import ModelCheckpoint

# Preprocess for ResNet50
X_tr_res  = res_preprocess(X_tr  * 255)
X_val_res = res_preprocess(X_val * 255)
X_te_res  = res_preprocess(X_test_norm * 255)

resnet_base = ResNet50(weights='imagenet', include_top=False, input_shape=(128,128,3))
for layer in resnet_base.layers:
    layer.trainable = False

resnet_model = Sequential([
    resnet_base,
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

res_train_gen = ImageDataGenerator(
    rotation_range=20, zoom_range=0.2,
    width_shift_range=0.2, height_shift_range=0.2,
    horizontal_flip=True
).flow(X_tr_res, y_tr, batch_size=32)

res_val_gen = ImageDataGenerator().flow(X_val_res, y_val, batch_size=32)

resnet_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=3,
                           restore_best_weights=True)
checkpoint = ModelCheckpoint(
    'best_resnet_model.h5',
    monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
)

print('🚀 Training ResNet50...')
history_resnet = resnet_model.fit(
    res_train_gen,
    validation_data=res_val_gen,
    epochs=15,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

plt.figure(figsize=(8,5))
plt.plot(history_resnet.history['accuracy'],     label='Train')
plt.plot(history_resnet.history['val_accuracy'], label='Val')
plt.title('ResNet50 Accuracy'); plt.xlabel('Epoch')
plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,5))
plt.plot(history_resnet.history['loss'],     label='Train')
plt.plot(history_resnet.history['val_loss'], label='Val')
plt.title('ResNet50 Loss'); plt.xlabel('Epoch')
plt.legend(); plt.tight_layout(); plt.show()

res_loss, res_acc = resnet_model.evaluate(X_te_res, y_test_cat, verbose=0)
print(f'\n✅ ResNet50 Accuracy: {res_acc:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 13 — Load best saved ResNet model
# ═══════════════════════════════════════════════════════════
from tensorflow.keras.models import load_model

best_model = load_model('best_resnet_model.h5')
best_loss, best_acc = best_model.evaluate(X_te_res, y_test_cat, verbose=0)
print(f'✅ Best Saved ResNet50 Accuracy: {best_acc:.4f}')

# Use best model accuracy for comparison
res_acc = best_acc

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 14 — Model comparison table (same as your Colab)
# ═══════════════════════════════════════════════════════════
from sklearn.metrics import f1_score
import pandas as pd

y_true = np.argmax(y_test_cat, axis=1)

# Basic CNN
basic_pred = np.argmax(model.predict(X_test_norm, verbose=0), axis=1)
basic_f1   = f1_score(y_true, basic_pred, average='weighted')

# Deep CNN
deep_pred = np.argmax(deep_cnn.predict(X_test_norm, verbose=0), axis=1)
deep_f1   = f1_score(y_true, deep_pred, average='weighted')

# VGG16
vgg_pred = np.argmax(vgg_model.predict(X_te_vgg, verbose=0), axis=1)
vgg_f1   = f1_score(y_true, vgg_pred, average='weighted')

# ResNet50
res_pred = np.argmax(best_model.predict(X_te_res, verbose=0), axis=1)
res_f1   = f1_score(y_true, res_pred, average='weighted')

comparison_df = pd.DataFrame({
    'Model':          ['CNN Basic', 'Deep CNN', 'ResNet50', 'VGG16'],
    'Test Accuracy':  [round(test_acc,4), round(deep_acc,4),
                       round(res_acc,4),  round(vgg_acc,4)],
    'F1 Score':       [round(basic_f1,4), round(deep_f1,4),
                       round(res_f1,4),   round(vgg_f1,4)],
    'Overfitting':    ['Yes', 'Reduced', 'Low', 'Low']
})

print(comparison_df.to_string(index=False))
comparison_df

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 15 — Confusion matrix for best model
# ═══════════════════════════════════════════════════════════
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_true, res_pred)

plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('ResNet50 — Confusion Matrix', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 16 — Predict a single image with best model
# ═══════════════════════════════════════════════════════════
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre

def predict_xray(img_path):
    """Usage: predict_xray('path/to/xray.jpg')"""
    raw = np.array(Image.open(str(img_path)).convert('RGB'))
    inp = np.array(Image.open(str(img_path)).convert('RGB')
                   .resize((IMG_SIZE, IMG_SIZE)))
    inp = res_pre(inp.astype('float32'))
    inp = np.expand_dims(inp, 0)
    proba = best_model.predict(inp, verbose=0)[0]
    pred  = classes[np.argmax(proba)]
    conf  = np.max(proba) * 100

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9,4))
    a1.imshow(raw); a1.axis('off'); a1.set_title('Input X-Ray')
    colors = ['#e74c3c' if c == pred else '#bdc3c7' for c in classes]
    a2.barh(classes, proba*100, color=colors)
    a2.set_xlabel('Confidence (%)')
    a2.set_title(f'Prediction: {pred} ({conf:.1f}%)', fontweight='bold')
    a2.set_xlim(0, 100)
    for i, v in enumerate(proba*100):
        a2.text(v+0.5, i, f'{v:.1f}%', va='center')
    plt.tight_layout(); plt.show()
    return pred, conf

# Demo with first test image
test_imgs = sorted(pathlib.Path(test_path).rglob('*.jpg')) + \
            sorted(pathlib.Path(test_path).rglob('*.jpeg'))
if test_imgs:
    p, c = predict_xray(test_imgs[0])
    print(f'\n🩺 Prediction: {p}  ({c:.1f}% confidence)')